In [20]:

import pandas as pd
import os


# Read the CSV file
data = pd.read_csv('sql_queries.csv')

# Display the data

total_queries = len(data)
print(total_queries)
data.head()


1676


,SQL Query
0,"No value found for ""SQL Query"" key"
1,"No value found for ""SQL Query"" key"
2,SELECT player_id FROM public.agg_season_player...
3,"No value found for ""SQL Query"" key"
4,"No value found for ""SQL Query"" key"


In [21]:
# Clean queries that have less than 10 characters

clean_data = data[data['SQL Query'].str.len() >= 10]

clean_data_len = len(clean_data)
print(f"Removed {total_queries - clean_data_len}")


Removed 5


In [22]:
clean_data2 = clean_data[clean_data['SQL Query'] != 'No value found for "SQL Query" key']
clean_data2_len = len(clean_data2)

print(f"Removed {clean_data_len - clean_data2_len}")


Removed 48


In [23]:
# Calculate the length of the DataFrame before removing duplicates
clean_data2_len = len(clean_data2)

# Remove duplicates without using inplace
rmv_dupli_data = clean_data2.drop_duplicates()

# Calculate the length of the DataFrame after removing duplicates
rmv_dupli_data_len = len(rmv_dupli_data)

# Print the number of duplicates removed
print(f"Removed {clean_data2_len - rmv_dupli_data_len} duplicates")

Removed 461 duplicates


In [25]:
len_data = len(rmv_dupli_data)

print(f"Rows: {len_data}")

rmv_dupli_data.to_csv('sql_queries_clean.csv', index=False)


Rows: 1162


# Running queries in DB

In [27]:

# Read the CSV file
queries_df = pd.read_csv('sql_queries_clean.csv')

queries_df.head()

,SQL Query
0,SELECT player_id FROM public.agg_season_player...
1,"SELECT player_name, SUM(dl_index) AS total_goa..."
2,"SELECT player_name, goals FROM vm_agg_season_p..."
3,"SELECT player_name, goals FROM vm_agg_season_p..."
4,"SELECT player_id, team_id, score_pd FROM publi..."


SQL Query    SELECT player_id FROM public.agg_season_player...
Name: 0, dtype: object

In [41]:
from logger_manager import LoggerManager
from run_query import get_db_data, init_connection_pool,close_connection_pool, get_db_data2
from print_colors import print_colored

def test_query(query, logger):
    data = get_db_data(query,logger)
    
    if data[0] == 'Database error':
        print("Failed query")
        return False, data[2]

    else:
        print("Sucessful query")
        columns , rows = data
        # Calculate the shape
        num_columns = len(columns)
        num_rows = len(rows)
        shape = (num_rows, num_columns)
        print(f"Data Shape: {shape} ")
        return True , shape
    
logger = LoggerManager.get_logger("SQL_Query_test2").get_logger()
logger.info("TEST 33")

query = queries_df.loc[0, 'SQL Query']
print(query)


init_connection_pool()

valid, description = test_query(query, logger)
print(valid, description)
close_connection_pool()



06-21 16:08:31 - logger_SQL_Query_test2 - INFO - TEST 33
INFO:logger_SQL_Query_test2:TEST 33


SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1
Database connection pool initialized at: 06-21 16:08:31
                                                                                


06-21 16:08:31 - logger_SQL_Query_test2 - INFO - Fetching data from the database at: 06-21 16:08:31 -> SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1
INFO:logger_SQL_Query_test2:Fetching data from the database at: 06-21 16:08:31 -> SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1
ERROR:root:Database error: column "age" does not exist
LINE 1: ...agg_season_player_pd WHERE season_id = 816527 AND age < 25 A...
                   

Failed query RED
False column "age" does not exist
LINE 1: ...agg_season_player_pd WHERE season_id = 816527 AND age < 25 A...
                                                             ^

Database connection pool closed at: 06-21 16:08:31
                                                                                


In [46]:
import time
import pandas as pd
from logger_manager import LoggerManager
from run_query import get_db_data, init_connection_pool, close_connection_pool
from print_colors import print_colored

def test_query(query, logger):
    data = get_db_data(query, logger)
    
    if data[0] == 'Database error':
        print("Failed query")
        return False, data[2]

    else:
        print("Successful query")
        columns, rows = data
        # Calculate the shape
        num_columns = len(columns)
        num_rows = len(rows)
        shape = (num_rows, num_columns)
        print(f"Data Shape: {shape} ")
        return True, shape

def run_queries_in_batches(queries_df, batch_size=20, wait_time=2):
    # Initialize the connection pool
    init_connection_pool()

    # Prepare results list
    results = []

    logger = LoggerManager.get_logger("SQL_Query_test2").get_logger()
    logger.info("Starting batch query test")

    # Process queries in batches
    for i in range(0, len(queries_df), batch_size):
        batch = queries_df.iloc[i:i + batch_size]

        for index, row in batch.iterrows():
            query = row['SQL Query']
            print(f"Running query: {query}")

            valid, description = test_query(query, logger)
            results.append({
                'SQL Query': query,
                'Valid': valid,
                'Description': description
            })
        

        # Wait between batches
        time.sleep(wait_time)

    # Close the connection pool
    close_connection_pool()

    # Create a new DataFrame from the results
    results_df = pd.DataFrame(results)
    return results_df

# Example usage:
# Assuming queries_df is your DataFrame containing SQL queries
# queries_df = pd.DataFrame({'SQL Query': [...your queries...]})

# Run the queries in batches
results_df = run_queries_in_batches(queries_df)

# Print or save the results DataFrame
print(results_df)
# Optionally, save to a CSV file
results_df.to_csv('sql_query_results.csv', index=True)

Database connection pool initialized at: 06-21 16:16:04
                                                                                


06-21 16:16:04 - logger_SQL_Query_test2 - INFO - Starting batch query test
INFO:logger_SQL_Query_test2:Starting batch query test


Running query: SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1


06-21 16:16:04 - logger_SQL_Query_test2 - INFO - Fetching data from the database at: 06-21 16:16:04 -> SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1
INFO:logger_SQL_Query_test2:Fetching data from the database at: 06-21 16:16:04 -> SELECT player_id FROM public.agg_season_player_pd WHERE season_id = 816527 AND age < 25 AND height > 180 AND team_id IN (4265, 4279) AND games_played_65mins >= 20 AND mins_played >= 2000 AND score_pd_float < 50 AND player_id IN (SELECT player_id FROM t_player WHERE country_id = 1 AND position = 'DL') ORDER BY score_pd DESC LIMIT 1
ERROR:root:Database error: column "age" does not exist
LINE 1: ...agg_season_player_pd WHERE season_id = 816527 AND age < 25 A...
                   

Failed query
Running query: SELECT player_name, SUM(dl_index) AS total_goals FROM vm_agg_season_player_similarity WHERE season_id = 816527 GROUP BY player_name ORDER BY total_goals DESC LIMIT 1


...

Successful query
Data Shape: (5, 3) 
Running query: SELECT player_id, team_id, season_id, player_name, passes_total FROM vm_agg_season_player_all WHERE season_id IN (816527, 816019, 815453) AND league_type_id = 1 ORDER BY passes_total DESC LIMIT 5


06-21 16:27:27 - logger_SQL_Query_test2 - INFO - Fetching data from the database at: 06-21 16:27:27 -> SELECT player_id, team_id, season_id, player_name, passes_total FROM vm_agg_season_player_all WHERE season_id IN (816527, 816019, 815453) AND league_type_id = 1 ORDER BY passes_total DESC LIMIT 5
INFO:logger_SQL_Query_test2:Fetching data from the database at: 06-21 16:27:27 -> SELECT player_id, team_id, season_id, player_name, passes_total FROM vm_agg_season_player_all WHERE season_id IN (816527, 816019, 815453) AND league_type_id = 1 ORDER BY passes_total DESC LIMIT 5
ERROR:root:Database error: relation "vm_agg_season_player_all" does not exist
LINE 1: ...eam_id, season_id, player_name, passes_total FROM vm_agg_sea...
                                                             ^



Failed query
Running query: SELECT player_id, passes_total, season_id FROM vm_agg_season_player_opta WHERE season_id IN (816527, 816019, 815453) ORDER BY passes_total DESC LIMIT 5


06-21 16:27:27 - logger_SQL_Query_test2 - INFO - Fetching data from the database at: 06-21 16:27:27 -> SELECT player_id, passes_total, season_id FROM vm_agg_season_player_opta WHERE season_id IN (816527, 816019, 815453) ORDER BY passes_total DESC LIMIT 5
INFO:logger_SQL_Query_test2:Fetching data from the database at: 06-21 16:27:27 -> SELECT player_id, passes_total, season_id FROM vm_agg_season_player_opta WHERE season_id IN (816527, 816019, 815453) ORDER BY passes_total DESC LIMIT 5


Successful query
Data Shape: (5, 3) 
Database connection pool closed at: 06-21 16:27:29
                                                                                
                                              SQL Query  Valid  \
0     SELECT player_id FROM public.agg_season_player...  False   
1     SELECT player_name, SUM(dl_index) AS total_goa...   True   
2     SELECT player_name, goals FROM vm_agg_season_p...   True   
3     SELECT player_name, goals FROM vm_agg_season_p...   True   
4     SELECT player_id, team_id, score_pd FROM publi...   True   
...                                                 ...    ...   
1157  SELECT player_id, player_name, team_id, season...   True   
1158  SELECT player_id, SUM(score_pd) AS goals_score...   True   
1159  SELECT player_id, passes_accurate_90, season_i...   True   
1160  SELECT player_id, team_id, season_id, player_n...  False   
1161  SELECT player_id, passes_total, season_id FROM...   True   

                                      